In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForMaskedLM, pipeline

model_name = "gerulata/slovakbert"

print(f"Loading {model_name}...")

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForMaskedLM.from_pretrained(model_name)

# Move to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

# Pipeline approach
# fill_mask_pipeline = pipeline("fill-mask", model=model, tokenizer=tokenizer, device=0 if torch.cuda.is_available() else -1)

# print("\n--- Fill Mask Example (Pipeline Way) ---")

# text_pipeline_1 = f"Je to veľmi pekný {tokenizer.mask_token}."
# prediction_pipeline_1 = fill_mask_pipeline(text_pipeline_1)
# print(f"\nInput: '{text_pipeline_1}'")
# print(f"Prediction: {prediction_pipeline_1}")

# text_pipeline_2 = f"Dnes je krásny {tokenizer.mask_token}."
# prediction_pipeline_2 = fill_mask_pipeline(text_pipeline_2)
# print(f"\nInput: '{text_pipeline_2}'")
# print(f"Prediction: {prediction_pipeline_2}")

print("\n--- Manual Fill Mask Example ---")

def get_manual_fill_mask(text, model, tokenizer, device, top_k=5):
    # 1. Tokenize the input
    inputs = tokenizer(
        text,
        return_tensors="pt",
        add_special_tokens=True,
        truncation=True,
        padding='max_length',
        max_length=512
    ).to(device)

    # 2. Perform model inference
    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits # [batch_size, sequence_length, vocab_size]

    # Find the position of the mask token
    mask_token_index = torch.where(inputs["input_ids"] == tokenizer.mask_token_id)[1]
    if mask_token_index.shape[0] == 0:
        return {"error": "No mask token found in the input text."}

    # Extract logits for the masked token (assuming a single mask token for simplicity)
    mask_logits = logits[0, mask_token_index[0], :]

    # Apply softmax to get probabilities
    probabilities = torch.softmax(mask_logits, dim=-1)

    # Get the top_k predicted tokens and their scores
    top_k_scores, top_k_indices = torch.topk(probabilities, top_k)

    predictions = []
    for score, index in zip(top_k_scores, top_k_indices):
        predictions.append(
            {
                "score": score.item(),
                "token": tokenizer.decode(index.item()),
                "token_id": index.item()
            }
        )
    return predictions

# Example 1: Simple sentence with one mask
text_manual_1 = f"Dnes je krásny {tokenizer.mask_token}."
prediction_manual_1 = get_manual_fill_mask(text_manual_1, model, tokenizer, device)
print(f"\nInput (manual way): '{text_manual_1}'")
print(f"Prediction (manual way): {prediction_manual_1}")

# Example 2: Another sentence with a mask
text_manual_2 = f"Chcem si prečítať dobrú {tokenizer.mask_token}."
prediction_manual_2 = get_manual_fill_mask(text_manual_2, model, tokenizer, device)
print(f"\nInput (manual way): '{text_manual_2}'")
print(f"Prediction (manual way): {prediction_manual_2}")

# Example 3: More complex sentence
text_manual_3 = f"Bol to veľmi zaujímavý {tokenizer.mask_token} o slovenskej histórii."
prediction_manual_3 = get_manual_fill_mask(text_manual_3, model, tokenizer, device)
print(f"\nInput (manual way): '{text_manual_3}'")
print(f"Prediction (manual way): {prediction_manual_3}")

Loading gerulata/slovakbert...


config.json:   0%|          | 0.00/581 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/772 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/203 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie lm_head.bias to lm_head.decoder.bias, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
RobertaForMaskedLM LOAD REPORT from: gerulata/slovakbert
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



--- Manual Fill Mask Example ---

Input (manual way): 'Dnes je krásny <mask>.'
Prediction (manual way): [{'score': 0.9183416366577148, 'token': ' deň', 'token_id': 364}, {'score': 0.023780815303325653, 'token': ' čas', 'token_id': 380}, {'score': 0.019899537786841393, 'token': ' večer', 'token_id': 1203}, {'score': 0.007965111173689365, 'token': ' den', 'token_id': 733}, {'score': 0.0027890210039913654, 'token': ' dátum', 'token_id': 7385}]

Input (manual way): 'Chcem si prečítať dobrú <mask>.'
Prediction (manual way): [{'score': 0.8696143627166748, 'token': ' knihu', 'token_id': 3417}, {'score': 0.05101681500673294, 'token': ' literatúru', 'token_id': 23837}, {'score': 0.03485553711652756, 'token': ' knižku', 'token_id': 32083}, {'score': 0.005327950231730938, 'token': ' klasiku', 'token_id': 43546}, {'score': 0.003412497229874134, 'token': ' vec', 'token_id': 778}]

Input (manual way): 'Bol to veľmi zaujímavý <mask> o slovenskej histórii.'
Prediction (manual way): [{'score': 0.35298

In [ ]:
!rm -rf ~/.cache/huggingface
print("Hugging Face cache cleared.")

Hugging Face cache cleared.
